# Exploratory Data Analysis: Prioritization of Early Churn Detection & Retention Strategies

## 1. Business Understanding
**Problem Statement:**
Develop a system that quantifies churn risk and integrates prioritization tools, allowing companies to identify at-risk customers and focus retention efforts where they will have the greatest impact.

**Key Business Questions:**

- Can we detect early signs of churn before a customer leaves?
- Which behavioral patterns indicate increased churn risk?
- Which customers should be prioritized for retention efforts?

**Business Objectives:**

- Identify customers at risk of churn before cancellation
- Generate churn probability scores (0–1)
- Enable businesses to prioritize retention efforts

**Success Criteria:**

- High recall (minimize missed churn cases)
- Interpretable model outputs
- Functional decision-support dashboard



## 2. Data Understanding

The dataset consists of merged user-level information including:
- Demographics (members_v3)
- Subscription transactions (transactions_v2)
- User listening behavior(user_logs_v2)
- Churn label (train_v2)

Each row represents a customer with aggregated behavioral and subscription features.

### 2.1 Data dictionary

#### Training Dataset

Churn is defined as no new subscription within 30 days after membership expiration

| Variable               | Meaning                                                                                                                     |
| ---------------------- | --------------------------------------------------------------------------------------------------------------------------- |
| **msno**               | Unique identifier                                                                            |
| **is_churn**             | Target variable indicating churn status.1 = Churn, 0 = Renewal. |


#### Testing Dataset

| Variable               | Meaning                                                                                                                     |
| ---------------------- | --------------------------------------------------------------------------------------------------------------------------- |
| **msno**               | Unique identifier                                                                            |
| **is_churn**             | Predicted churn label to be filled by the model. |


#### Transaction Dataset

| Variable                   | Meaning                                                                         |
| -------------------------- | ------------------------------------------------------------------------------- |
| **msno**                   | Unique user identifier linking subscription activity to a specific user.        |
| **payment_method_id**      | Identifier representing the payment method used by the user.                    |
| **payment_plan_days**      | Number of days included in the subscription plan purchased by the user.         |
| **plan_list_price**        | The standard listed price of the subscription plan in New Taiwan Dollars (NTD). |
| **actual_amount_paid**     | The actual amount the user paid after discounts or promotions.                  |
| **is_auto_renew**          | Indicates whether the user enabled automatic subscription renewal.              |
| **transaction_date**       | The date on which the subscription transaction occurred.                        |
| **membership_expire_date** | The date when the user's subscription membership is scheduled to expire.        |
| **is_cancel**              | Indicates whether the user canceled their subscription in that transaction.     |


#### User Listening Logs Dataset

| Variable       | Meaning                                                                                      |
| -------------- | -------------------------------------------------------------------------------------------- |
| **msno**       | Unique user identifier.                                                                      |
| **date**       | Date when the listening activity occurred.                                                   |
| **num_25**     | Number of songs played for less than 25% of their total length.                              |
| **num_50**     | Number of songs played between 25% and 50% of their total length.                            |
| **num_75**     | Number of songs played between 50% and 75% of their total length.                            |
| **num_985**    | Number of songs played between 75% and 98.5% of their total length.                          |
| **num_100**    | Number of songs played for more than 98.5% of their total length (essentially fully played). |
| **num_unq**    | Number of unique songs played by the user.                                                   |
| **total_secs** | Total number of seconds the user spent listening to music.                                   |


### 2.2 Dataset Sizes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:

training = pd.read_csv("train_v2.csv")
test = pd.read_csv("test_v2.csv")
transactions = pd.read_csv("transactions_v2.csv")
user_logs = pd.read_csv("user_logs_v2.csv")
members = pd.read_csv("members_v3.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
print("Training Shape:", training.shape)
training.head()

Training Shape: (970960, 2)


,msno,is_churn
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1


In [7]:
print("Testing Shape:", test.shape)
test.head()

Testing Shape: (907471, 2)


,msno,is_churn
0,4n+fXlyJvfQnTeKXTWT507Ll4JVYGrOC8LHCfwBmPE4=,0
1,aNmbC1GvFUxQyQUidCVmfbQ0YeCuwkPzEdQ0RwWyeZM=,0
2,rFC9eSG/tMuzpre6cwcMLZHEYM89xY02qcz7HL4//jc=,0
3,WZ59dLyrQcE7ft06MZ5dj40BnlYQY7PHgg/54+HaCSE=,0
4,aky/Iv8hMp1/V/yQHLtaVuEmmAxkB5GuasQZePJ7NU4=,0


In [8]:
print("Transactions Shape:", transactions.shape)
transactions.head()

Transactions Shape: (21547746, 9)


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=,41,30,129,129,1,20150930,20151101,0
1,AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=,41,30,149,149,1,20150930,20151031,0
2,UkDFI97Qb6+s2LWcijVVv4rMAsORbVDT2wNXF0aVbns=,41,30,129,129,1,20150930,20160427,0
3,M1C56ijxozNaGD0t2h68PnH2xtx5iO5iR2MVYQB6nBI=,39,30,149,149,1,20150930,20151128,0
4,yvj6zyBUaqdbUQSrKsrZ+xNDVM62knauSZJzakS9OW4=,39,30,149,149,1,20150930,20151121,0


In [9]:

print("User Logs Shape:", user_logs.shape)
user_logs.head()

User Logs Shape: (18396362, 9)


,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
0,u9E91QDTvHLq6NXjEaWv8u4QIqhrHk72kE+w31Gnhdg=,20170331,8,4,0,1,21,18,6309.273
1,nTeWW/eOZA/UHKdD5L7DEqKKFTjaAj3ALLPoAWsU8n0=,20170330,2,2,1,0,9,11,2390.699
2,2UqkWXwZbIjs03dHLU9KHJNNEvEkZVzm69f3jCS+uLI=,20170331,52,3,5,3,84,110,23203.337
3,ycwLc+m2O0a85jSLALtr941AaZt9ai8Qwlg9n0Nql5U=,20170331,176,4,2,2,19,191,7100.454
4,EGcbTofOSOkMmQyN1NMLxHEXJ1yV3t/JdhGwQ9wXjnI=,20170331,2,1,0,1,112,93,28401.558


In [10]:
print("Members Shape:", members.shape)
members.head()

Members Shape: (6769473, 6)


,msno,city,bd,gender,registered_via,registration_init_time
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NaN,11,20110911
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NaN,7,20110914
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NaN,11,20110915
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NaN,11,20110915
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,20110915


# 3. Data Preperation

### 3.1 Data Cleaning

3.1.1 Training Dataset

In [11]:
# Missing values
missing_train = training.isnull().sum()
print("Missing values:\n", missing_train)

# Duplicate users
train_duplicates = training['msno'].duplicated().sum()
print("Duplicate users:", train_duplicates)

training = training.rename(columns={'is_churn': 'target'})

Missing values:
 msno        0
is_churn    0
dtype: int64
Duplicate users: 0


3.1.2 Testing Dataset

In [12]:
# Missing values

missing_test = test.isnull().sum()
print("Missing values:\n", missing_test)

# Duplicate users
test_duplicates = test['msno'].duplicated().sum()
print("Duplicate users:", test_duplicates)

Missing values:
 msno        0
is_churn    0
dtype: int64
Duplicate users: 0


**Key Observation:**

- No missing values were found
- No duplicate users were detected
- Dataset already clean and consistent

3.1.3 Transaction Dataset

In [13]:
# Missing values
missing_transactions = transactions.isnull().sum()
print("Missing values:\n", missing_transactions)

# Duplicate rows
transactions_duplicates = transactions.duplicated().sum()
print("Exact duplicate rows:", transactions_duplicates)


Missing values:
 msno                      0
payment_method_id         0
payment_plan_days         0
plan_list_price           0
actual_amount_paid        0
is_auto_renew             0
transaction_date          0
membership_expire_date    0
is_cancel                 0
dtype: int64
Exact duplicate rows: 3339


In [14]:
# Check for negative or zero values
print("Invalid plan days:", (transactions['payment_plan_days'] <= 0).sum())
print("Invalid prices:", (transactions['plan_list_price'] < 0).sum())
print("Invalid payments:", (transactions['actual_amount_paid'] < 0).sum())

Invalid plan days: 870124
Invalid prices: 0
Invalid payments: 0


**Key Observation:**

- No missing values were found
- No duplicate users were detected
- No negative payments or price values
- There are 2,218 records with invalid plan days.

3.1.4 User Log Dataset

In [15]:
# Missing values
missing_user_logs = user_logs.isnull().sum()
print("Missing values:\n", missing_user_logs)

# Duplicate rows
user_logs_duplicates = user_logs.duplicated().sum()
print("Duplicate rows:", user_logs_duplicates)

Missing values:
 msno          0
date          0
num_25        0
num_50        0
num_75        0
num_985       0
num_100       0
num_unq       0
total_secs    0
dtype: int64
Duplicate rows: 0


In [16]:
print("Negative listening time:", (user_logs['total_secs'] < 0).sum())

# Check extreme values
print("Very large listening time (>24h):", (user_logs['total_secs'] > 86400).sum())

Negative listening time: 0
Very large listening time (>24h): 4200


**Key Observation:**

- No missing values were found
- No duplicate users were detected
- No negative listening times, however, the dataset did contain 4200 extreme values

3.1.5 Memeber Dataset

In [17]:
# Missing values
missing_members = members.isnull().sum()
print("Missing values:\n", missing_members)

# Duplicate users
members_duplicates = members['msno'].duplicated().sum()
print("Duplicate users:", members_duplicates)

Missing values:
 msno                            0
city                            0
bd                              0
gender                    4429505
registered_via                  0
registration_init_time          0
dtype: int64
Duplicate users: 0


In [18]:
# Check age distribution issues
print("Age <= 0:", (members['bd'] <= 0).sum())
print("Age < 10:", ((members['bd'] > 0) & (members['bd'] < 10)).sum())
print("Age > 100:", (members['bd'] > 100).sum())

Age <= 0: 4540489
Age < 10: 899
Age > 100: 5377


**Key Observations:**
- Gender contains 4,429,505 missing values
- No duplicate users detected
- Age data is severely unreliable, with a large number of invalid values (≤ 0).


### 3.2 Data Transformation and Feature Standardization

3.2.1 Training Dataset

In [19]:
training.columns = training.columns.str.strip().str.lower()
print("Standardized column names:", training.columns)


Standardized column names: Index(['msno', 'target'], dtype='object')


The training dataset did not require heavy transformation. Column names were standardized by removing extra spaces and converting them to lowercase to ensure consistency across all merged datasets.

3.2.2 Testing Dataset

In [20]:
test.columns = test.columns.str.strip().str.lower()
print("Standardized column names:", test.columns)

Standardized column names: Index(['msno', 'is_churn'], dtype='object')


The training dataset did not require heavy transformation. Column names were standardized by removing extra spaces and converting them to lowercase to ensure consistency across all merged datasets.

3.2.3 Transaction Dataset

In [21]:

transactions.columns = transactions.columns.str.strip().str.lower()
transactions = transactions.drop_duplicates()
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], format='%Y%m%d')
transactions['membership_expire_date'] = pd.to_datetime(transactions['membership_expire_date'], format='%Y%m%d')

transactions['payment_plan_days'] = transactions['payment_plan_days'].replace(0, np.nan)

exchange_rate = 0.031

transactions['plan_price_usd'] = transactions['plan_list_price'] * exchange_rate
transactions['amount_paid_usd'] = transactions['actual_amount_paid'] * exchange_rate



Transaction dataset was standardized to improve interpretability and prepare them for aggregation. Dates were converted to datetime format so that time-based subscription measures could be calculated. When checking for extreme values it payment_plan_days has over 2000 as a result it was replaced with zero. Variables such as plan_list_price and actual_amount_paid covert into USD as the were orginally in New Taiwan Dollar, however both versions were kept for trace ability.

3.2.4 User Log Dataset

In [22]:
user_logs.columns = user_logs.columns.str.strip().str.lower()

user_logs['date'] = pd.to_datetime(user_logs['date'], format='%Y%m%d')


user_logs['total_secs'] = user_logs['total_secs'] <= 86400


User logs were standardized and the date column was converted to datetime format for consistency. A small number of observations contained extremely large listening times, total_secs was capped at 24 hours to remain realistic.

3.2.5 Member Dataset

In [23]:
members.columns = members.columns.str.strip().str.lower()

members['registration_init_time'] = pd.to_datetime(members['registration_init_time'], format='%Y%m%d')

members.loc[(members['bd'] <= 10) | (members['bd'] > 100), 'bd'] = np.nan

members['gender'] = members['gender'].fillna('Prefer not to say')



Member column names were standardized and registration dates were converted into datetime format. The bd variable contained  unrealistic values such as zero, negative ages, ages below 10, and ages above 100 were treated as invalid and replaced with missing values. Gender also had an extreme amount of missing data and was filled in using a place filler.

### 3.3 Feature Engineering

3.3.1 Transaction-Based Features

In [ ]:
transactions['days_to_expire'] = (transactions['membership_expire_date'] - transactions['transaction_date']).dt.days

transactions['discount'] = transactions['plan_price_usd'] - transactions['amount_paid_usd']

transactions['gap_days'] = transactions.groupby('msno')['transaction_date'].diff().dt.days

transactions['discount_pct'] = np.where(
    transactions['plan_price_usd'] > 0,
    transactions['discount'] / transactions['plan_price_usd'],
    0
)
transactions['used_discount'] = (transactions['discount'] > 0).astype(int)
transactions_agg = transactions.groupby('msno').agg(
    transaction_count=('msno', 'size'),
    avg_payment_method=('payment_method_id', 'mean'),
    avg_plan_days=('payment_plan_days', 'mean'),
    total_plan_list_price=('plan_list_price', 'sum'),
    total_actual_paid=('actual_amount_paid', 'sum'),
    avg_actual_paid=('actual_amount_paid', 'mean'),
    avg_discount=('discount', 'mean'),
    renewal_rate=('is_auto_renew', 'mean'),
    cancel_count=('is_cancel', 'sum'),
    avg_days_to_expire=('days_to_expire', 'mean'),
    max_days_to_expire=('days_to_expire', 'max'),
    first_transaction=("transaction_date", "min"),
    last_transaction=("transaction_date", "max"),
    last_expire=("membership_expire_date", "max"),
     avg_subscription_length=("subscription_length", "mean"),
    total_subscription_length=("subscription_length", "sum")
).reset_index()

def plan_category(days):
    if days <= 0:
        return 'Invalid'
    elif days <= 30:
        return 'Monthly'
    elif days <= 90:
        return 'Quarterly'
    elif days <= 183:
        return 'Semi_Annual'
    elif days <= 365:
        return 'Annual'
    return 'Extended'

transactions_agg['plan_category'] = transactions['payment_plan_days'].apply(plan_category)


Transaction records were aggregated at the user level to summarize subscription behavior. The goal of these features was to capture how users pay, how long their plans last, whether they renew automatically, and whether they have a history of cancellations.

3.3.2 User Engagement Features

In [ ]:
user_logs['total_songs'] = user_logs[['num_25','num_50','num_75','num_985','num_100']].sum(axis=1)



user_logs['avg_secs_per_song'] = user_logs['total_secs'] / user_logs['total_songs']

user_logs_agg = user_logs.groupby("msno", as_index=False).agg(
    total_listening_time=("total_secs", "sum"),
    avg_daily_usage=("total_secs", "mean"),
    total_unique_songs=("num_unq", "sum"),
    total_sessions=("total_plays", "sum"),
    num_log_days=("date", "nunique"),
    last_log_date=("date", "max"),
    num_25_sum=("num_25", "sum"),
    num_100_sum=("num_100", "sum")
)

user_logs_agg['completion_rate'] = user_logs['num_100'] / user_logs['total_songs']

user_logs_agg['skip_rate'] = user_logs['num_25'] / user_logs['total_songs']

User logs were also aggregated to summarize engagement patterns. Summary statistics were calculated to capture total usage, average daily activity, listening variability, and song diversity.

3.3.3 Memeber Features

In [ ]:

members['account_age_days'] = (pd.to_datetime('2017-02-28') - members['registration_init_time']).dt.days

def Loyalty_status(account_age_days):
    if account_age_days < 30:
        return 'New'
    elif account_age_days < 90:
        return 'Established'
    elif account_age_days < 365:
        return 'Loyal'
    elif account_age_days >= 730:
        return 'Veteran'
    else:
        return 'Invalid'

members['Loyalty_Status'] = members['account_age_days'].apply(Loyalty_status)

def Age_Group(bd):
    if bd < 10:
        return 'Child'
    elif bd < 18:
        return 'Teen'
    elif bd < 30:
        return 'Young_Adult'
    elif bd < 50:
        return 'Adult'
    elif bd < 70:
        return 'Middle_Age'
    elif bd <= 100:
        return 'Senior'
    else:
        return 'Invalid'

members['Age_Group'] = members['bd'].apply(Age_Group)


tier1_cities = [1, 5, 13, 4, 22]
tier2_cities = [15, 6, 14, 12, 9]

def city_tier(city):
    if city in tier1_cities:
        return 'Tier1'
    elif city in tier2_cities:
        return 'Tier2'
    else:
        return 'Tier3'

members['City_Tier'] = members['city'].apply(city_tier)


To enhance interpretability, several categorical features were engineered. Users were grouped into loyalty segments based on account age, capturing differences between new and long-term users. Age values were categorized into meaningful demographic groups after removing invalid entries. Additionally, cities were grouped into tiers to reflect potential differences in user behavior across regions.

In [ ]:
time_agg = transactions_agg[[
    "msno", "first_transaction", "last_transaction", "last_expire"
]].copy()

time_agg = time_agg.merge(
    members[["msno", "registration_init_time"]],
    on="msno",
    how="left"
)
time_agg = time_agg.merge(
    user_logs_agg[["msno", "last_log_date"]],
    on="msno",
    how="left"
)

reference_date = pd.to_datetime("2017-03-01")

time_agg = time_agg.groupby("msno", as_index=False).agg(
    first_transaction=("first_transaction", "min"),
    last_transaction=("last_transaction", "max"),
    last_expire=("last_expire", "max"),
    registration_init_time=("registration_init_time", "first"),
    last_log_date=("last_log_date", "max")
)

time_agg["membership_duration"] = (time_agg["last_expire"] - time_agg["first_transaction"]).dt.days
time_agg["days_since_last_transaction"] = (reference_date - time_agg["last_transaction"]).dt.days
time_agg["time_until_expiration"] = (time_agg["last_expire"] - reference_date).dt.days
time_agg["account_age_days"] = (reference_date - time_agg["registration_init_time"]).dt.days
time_agg["days_since_last_log"] = (reference_date - time_agg["last_log_date"]).dt.days

### 3.4 Data Integration
The datasets was integrated using msno, which uniquely identifies each user. The completed dataset was then saved as kkbox_dataset.csv for further analysis and modeling.

In [ ]:
kkbox_dataset_training = train.merge(transactions_agg, on='msno', how='left')
kkbox_dataset_training = kkbox_dataset_training.merge(members, on='msno', how='left')
kkbox_dataset_training = kkbox_dataset_training.merge(user_logs_agg, on='msno', how='left')
kkbox_dataset_training = kkbox_dataset_training.merge(time_agg, on='msno', how='left')

In [ ]:
kkbox_dataset_training = kkbox_dataset_training.drop_duplicates(subset='msno', keep='first')
kkbox_dataset_training.to_csv("kkbox_dataset_training.csv", index=False)

In [ ]:
kkbox_dataset_testing = test.merge(transactions_agg, on='msno', how='left')
kkbox_dataset_testing = kkbox_dataset_testing.merge(members, on='msno', how='left')
kkbox_dataset_testing = kkbox_dataset_testing.merge(time_agg, on='msno', how='left')
kkbox_dataset_testing = kkbox_dataset_testing.merge(user_logs_agg, on='msno', how='left')

In [ ]:
kkbox_dataset_training = kkbox_dataset_testing.drop_duplicates(subset='msno', keep='first')
kkbox_dataset_testing.to_csv("kkbox_dataset_testing.csv", index=False)

In [ ]:
df = pd.read_csv("kkbox_dataset_training.csv")
print("training dataset shape:", df.shape)
print("testing dataset shape:",pd.read_csv("kkbox_dataset_testing.csv").shape)

### 3.5 Feature Selection

3.5.1 Target Variable

In [ ]:
churn_counts = df['target'].value_counts().sort_index()
churn_pct = df['target'].value_counts(normalize=True).sort_index() * 100

churn_summary = pd.DataFrame({
    'Count': churn_counts,
    'Percentage': churn_pct.round(2)
})

print("Churn Distribution:")
print(churn_summary)
print()

In [ ]:
plt.subplots(figsize=(14, 5))
churn_counts.plot(kind='bar',  color=['#2ecc71', '#e74c3c'])
plt.title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
plt.xlabel('is_churn')
plt.ylabel('Count')
plt.xticks([0, 1], ['Retained (0)', 'Churned (1)'], rotation=0)


In [ ]:
retained_count = churn_counts.get(0, 0)
churned_count = churn_counts.get(1, 0)
print("Target Summary:")
print(f"Retained users (0): {retained_count:,}")
print(f"Churned users (1): {churned_count:,}")

if churned_count > 0:
    imbalance_ratio = retained_count / churned_count
    print(f"Majority-to-minority ratio: {imbalance_ratio:.1f}:1")

    if imbalance_ratio > 5:
        print("Status: Highly imbalanced ")
    elif imbalance_ratio > 2:
        print("Status: Moderately imbalanced ")
    else:
        print("Status: Fairly balanced")
else:
    print("Status: No churned samples found in the dataset")

3.5.2 Transaction- Based Features Analysis

In [ ]:
sns.boxplot(data=df, x='target', y='renewal_rate')
plt.title('Renewal Rate by Churn')
plt.xlabel('Target')
plt.ylabel('Renewal Rate')
plt.show()

In [ ]:
sns.boxplot(data=df, x='target', y='cancel_count')
plt.title('Cancel Count by Churn')
plt.xlabel('Target')
plt.ylabel('Cancel Count')
plt.show()

In [ ]:
sns.boxplot(data=df, x='target', y='transaction_count')
plt.title('Transaction Count by Churn')
plt.xlabel('Target')
plt.ylabel('Transaction Count')
plt.show()

In [ ]:
sns.boxplot(data=df, x='target', y='discount_usage')
plt.title('Discount Usage Rate by Churn')
plt.xlabel('Target')
plt.ylabel('Discount Usage Rate')
plt.show()

In [ ]:
sns.countplot(data=df, x='plan_category', hue='target')

plt.title('Plan Category vs Churn')
plt.xlabel('Plan Category')
plt.ylabel('User Count')
plt.xticks(rotation=45)

plt.show()

3.5.3 User Engagement Features Analysis

In [ ]:
sns.boxplot(data=df, x='target', y='total_secs')
plt.title('Total Listening Time by Churn')
plt.xlabel('Target')
plt.ylabel('Average Listening Time (secs)')
plt.show()

In [ ]:
sns.boxplot(data=df, x='target', y='completion_rate')
plt.title('Completion Rate by Churn')
plt.xlabel('Target')
plt.ylabel('Completion Rate')
plt.show()

In [ ]:
sns.boxplot(data=df, x='target', y='skip_rate')
plt.title('Skip Rate by Churn')
plt.xlabel('Target')
plt.ylabel('Skip Rate')
plt.show()

In [ ]:
sns.boxplot(data=df, x='target', y='total_songs')
plt.title('Total Songs Played by Churn')
plt.xlabel('Target')
plt.ylabel('Average Songs Played')
plt.show()

3.5.4 Demographic Features Analysis

In [ ]:
sns.countplot(data=df, x='Loyalty_Status', hue='target')
plt.title('Loyalty Status vs Churn')
plt.xlabel('Loyalty Status')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.boxplot(data=df, x='target', y='account_age_days')
plt.title('Account Age by Churn')
plt.xlabel('Target')
plt.ylabel('Account Age (Days)')
plt.show()

In [ ]:
sns.countplot(data=df, x='Age_Group', hue='target')
plt.title('Age Group vs Churn')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='City_Tier', hue='target')
plt.title('City Tier vs Churn')
plt.xlabel('City Tier')
plt.ylabel('Count')
plt.show()

3.5.5 Multivariate Analysis

In [ ]:
corr = df.select_dtypes(include='number').corr()

sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.5)

plt.title('Correlation Heatmap of Features')
plt.show()

In [ ]:
corr_target = df.select_dtypes(include='number').corr()['target']\
                .sort_values(ascending=False)

plt.figure(figsize=(8,6))
corr_target.drop('target').plot(kind='bar')

plt.title('Feature Correlation with Churn')
plt.ylabel('Correlation')
plt.show()

In [ ]:
top_features = corr_target.abs().sort_values(ascending=False)[1:15].index

plt.figure(figsize=(10,6))
sns.heatmap(df[top_features].corr(), cmap='coolwarm', annot=True)

plt.title('Top Feature Correlations')
plt.show()

## 4. Key Insights

- The target variable `is_churn` is imbalanced, with far fewer churned users than non-churned users. Because of this, accuracy alone is not a reliable evaluation metric.
- User behavior and subscription history appear to be more informative than demographic variables for predicting churn.
- Recency-based features such as `days_since_last_transaction` are likely strong predictors because inactive users are more likely to churn.
- Engagement features such as listening time, daily usage, skip rate, and completion rate can help identify users who are becoming less active.
- Subscription-related variables such as renewal frequency, average subscription length, and cancellation history are likely important indicators of churn risk.
- High-risk group: Customers receiving very large discounts (0.8–1.0) have the highest churn (~75%).Low discount users are more stable: Customers with little or no discount tend to stay longer.

## 5. Avoiding Data Leakage

Data leakage happens when the model uses information that would not have been available at the time the prediction is made. This can cause the model to perform unrealistically well during training and evaluation.

In this project, leakage may occur if features are created using future transactions, future expiration dates, or future user activity. For example, variables such as `last_transaction`, `last_expire`, `last_log_date`, `time_until_expiration`, and `days_since_last_transaction` can leak information if they are calculated using records that occur after the prediction date.

To avoid data leakage:

- Only use data available up to the prediction cutoff date when creating features.
- Define churn using a future label window that begins after the cutoff date.
- Do not allow the feature window and label window to overlap.
- Use time-based train/validation/test splits instead of random splitting.
- Fit preprocessing steps such as scaling, encoding, and imputation on the training set only.
- Exclude features that are directly derived from the target variable or contain future information.

A good rule is: if the value would not have been known at the time of prediction, it should not be used as a feature.